<div style="text-align: center; padding-top: 12%;">

# Backpropagation

## DATA 301 @ WM

<figure class="no-fragment">
<center>
<a href="https://github.com/yanhai-xiong/DATA301-AML/blob/main/Module%204%20-%20Introduction%20to%20Neural%20Networks/Section%201.2%20-%20Backpropagation.ipynb" target="_blank">
    <img class="no-fragment"
         src="https://brand.github.com/_next/static/media/logo-03.cc5e5332.png"
         width="80px"/>
</a>
</center>
</figure>
</div>

## <font color='blue' size=5pt>1. Our Example: A 2–3–1 Neural Network</font>

We use the **XOR classification example** from the coding section: two input features, three hidden neurons, and one output neuron.

<img src ="https://www.ritchievink.com/img/post-9-mlp/nn_diagram_1.png" width = 250>

- **Input → Hidden Layer:**
$
W_1 =
\begin{bmatrix}
w_{11}^{(1)} & w_{12}^{(1)} & w_{13}^{(1)} \\
w_{21}^{(1)} & w_{22}^{(1)} & w_{23}^{(1)}
\end{bmatrix}
\in \mathbb{R}^{2\times3},
\qquad
b_1 =
\begin{bmatrix}
b_1^{(1)} & b_2^{(1)} & b_3^{(1)}
\end{bmatrix}
\in \mathbb{R}^{1\times3}
$
- **Hidden → Output Layer:**
$
W_2 =
\begin{bmatrix}
w_{11}^{(2)} \\
w_{21}^{(2)} \\
w_{31}^{(2)}
\end{bmatrix}
\in \mathbb{R}^{3\times1},
\qquad
b_2 =
\begin{bmatrix}
b_1^{(2)}
\end{bmatrix}
\in \mathbb{R}^{1\times1}
$
- Every neuron applies the **sigmoid** activation $\sigma(z)=1/(1+e^{-z})$.

**Question:** How can we adjust *every* weight to reduce the prediction error?

## <font color='blue' size=5pt>2. Forward Propagation: Input → Prediction</font>

For $m$ samples, $X\in\mathbb{R}^{m\times2}$:

**Hidden layer**
$$
Z_1=XW_1+b_1,\qquad A_1=\sigma(Z_1)
$$

**Output layer**
$$
Z_2=A_1W_2+b_2,\qquad A_2=\sigma(Z_2)=\hat Y
$$

```python
network.z1 = X @ network.W1 + network.b1
network.a1 = network.sigmoid(network.z1)
network.z2 = network.a1 @ network.W2 + network.b2
network.a2 = network.sigmoid(network.z2)
```

**Shapes:** $X:(m,2)$ → $A_1:(m,3)$ → $A_2:(m,1)$.

**Key idea:** Each layer combines its inputs linearly, then applies a nonlinear activation.

## <font color='blue' size=5pt>3. Compute the Loss: How Wrong Are We?</font>

For the XOR example, targets are $y\in\{0,1\}$, and the sigmoid output is $\hat y\in(0,1)$.

For example, we use **half mean squared error**:
$$
\boxed{L=\frac{1}{2m}\sum_{i=1}^{m}(y_i-\hat y_i)^2}
$$

For **one illustrative sample**, if $y=1$ and $\hat y=0.8$:
$$L=\tfrac12(1-0.8)^2=0.02.$$

```python
loss = 0.5 * np.mean((y_true - y_pred) ** 2)
```
- The square makes positive and negative errors contribute equally.
- The factor $1/2$ cancels the $2$ when differentiating.
- The mean averages the loss over $m$ samples.

**Goal:** Find how much $L$ changes when each weight or bias changes.

## <font color='blue' size=5pt>4. Backpropagation: The Chain Rule</font>

The loss depends on earlier weights **through a chain of computations**:

$$W_1\;\longrightarrow\;Z_1\;\longrightarrow\;A_1\longrightarrow\;W_2\longrightarrow\;Z_2\;\longrightarrow\;A_2\;\longrightarrow\;L$$

To measure the effect of an output-layer weight $W_2$, apply the **chain rule**:
$$
\frac{\partial L}{\partial W_2}
=\frac{\partial L}{\partial A_2}
\frac{\partial A_2}{\partial Z_2}
\frac{\partial Z_2}{\partial W_2}
$$

For a hidden-layer weight $W_1$, the chain is longer: it also passes through $A_1$ and $Z_1$.

**Key idea:** Reuse gradients computed at later layers to efficiently compute gradients at earlier layers.

## <font color='blue' size=5pt>5. Output Layer: $\delta_2$, $dW_2$, $db_2$</font>

For each sample, $\ell_i=\tfrac12(y_i-a_{2,i})^2$ and $a_{2,i}=\sigma(z_{2,i})$.

**Step A: Error signal** (chain rule)
$$
\delta_2=\frac{\partial\ell}{\partial Z_2}
=\underbrace{(A_2-Y)}_{\text{prediction error}}\odot
\underbrace{A_2(1-A_2)}_{\sigma'(Z_2)}
$$

```python
delta2 = -(y_true - network.a2) * network.sigmoid_derivative(network.a2)
```

**Step B: Weight and bias gradients**, averaged over $m$ samples (because we are minimizing the average prediction loss, not the sum of prediction losses):
$$
\boxed{dW_2=\frac{A_1^T\delta_2}{m}},\qquad
\boxed{db_2=\frac{1}{m}\sum_{i=1}^m\delta_{2,i}}
$$

```python
self.gradients['dW2'] = (network.a1.T @ delta2) / m
self.gradients['db2'] = np.sum(delta2, axis=0, keepdims=True) / m
```

**Shapes:** $\delta_2:(m,1)$; $dW_2:(3,1)$; $db_2:(1,1)$.

## <font color='blue' size=5pt>6. Hidden Layer: $\delta_1$, $dW_1$, $db_1$</font>

The hidden layer has no direct target. Its error comes from the **next layer**.

**Step A: Propagate the error backward** through $W_2$ and the sigmoid:
$$
\boxed{\delta_1=(\delta_2W_2^T)\odot A_1(1-A_1)}
$$

```python
delta1 = (delta2 @ network.W2.T) * network.sigmoid_derivative(network.a1)
```

**Step B: Compute gradients** using the inputs to this layer:
$$
\boxed{dW_1=\frac{X^T\delta_1}{m}},\qquad
\boxed{db_1=\frac{1}{m}\sum_{i=1}^m\delta_{1,i}}
$$

```python
self.gradients['dW1'] = (X.T @ delta1) / m
self.gradients['db1'] = np.sum(delta1, axis=0, keepdims=True) / m
```

**Shapes:** $\delta_1:(m,3)$; $dW_1:(2,3)$; $db_1:(1,3)$.

**Key idea:** The transpose $W_2^T$ carries error signals from the output back to the hidden neurons.

## <font color='blue' size=5pt>7. Gradient Descent: Update the Parameters</font>

Backpropagation **computes gradients**; the optimizer **uses them** to update parameters.

For learning rate $\eta>0$:
$$
\begin{aligned}
W_1 &\leftarrow W_1-\eta\,dW_1 & b_1 &\leftarrow b_1-\eta\,db_1\\
W_2 &\leftarrow W_2-\eta\,dW_2 & b_2 &\leftarrow b_2-\eta\,db_2
\end{aligned}
$$

```python
network.W1 -= learning_rate * gradients['dW1']
network.b1 -= learning_rate * gradients['db1']
network.W2 -= learning_rate * gradients['dW2']
network.b2 -= learning_rate * gradients['db2']
```
- A **positive gradient** means increasing that parameter locally increases the loss, so move it downward.
- A **negative gradient** means increasing that parameter locally decreases the loss, so move it upward.
- Repeat forward pass → loss → backward pass → update over many epochs.

**Note:** The notebook's XOR training uses all four samples per update, so it is **full-batch gradient descent**, even though the optimizer class is named `SGDOptimizer`.

## <font color='blue' size=5pt>8. Summary: One Training Iteration</font>

| Stage | What happens? | Main quantities |
|:--|:--|:--|
| **Forward** | Input → hidden → output | $Z_1,A_1,Z_2,A_2$ |
| **Loss** | Compare prediction with target | $L=\frac{1}{2m}\sum(Y-A_2)^2$ |
| **Backward: output** | Compute output error and gradients | $\delta_2,dW_2,db_2$ |
| **Backward: hidden** | Propagate error and compute gradients | $\delta_1,dW_1,db_1$ |
| **Update** | Move parameters against gradients | $W_j,b_j\leftarrow W_j,b_j-\eta\nabla L$ |

$$
\underbrace{X\to A_1\to A_2\to L}_{\text{forward}}
\quad\longrightarrow\quad
\underbrace{\delta_2\to\delta_1\to\nabla L}_{\text{backward}}
\quad\longrightarrow\quad\text{update}
$$

**Remember:** Backpropagation is an efficient application of the **chain rule**, not a separate optimization algorithm. Gradient descent performs the updates.

**Check your understanding:** Why does $dW_2$ use $A_1^T$, while $dW_1$ uses $X^T$?

# Coding Application 
Backpropagation Implementation from Scratch - Neural Network for XOR Problem

This notebook implements a simple 2-layer neural network with backpropagation to solve the XOR problem.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

## 1. Neural Network Class

Implements a 2-layer neural network with sigmoid activation function.

In [2]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        # Initialize weights with small random values
        self.W1 = np.random.randn(input_size, hidden_size) * 0.5
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.5
        self.b2 = np.zeros((1, output_size))
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def sigmoid_derivative(self, a):
        return a * (1 - a)
    
    def forward(self, X):
        # Hidden layer
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.sigmoid(self.z1)
        # Output layer
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

## 2. MSE Loss Class

Mean Squared Error loss with backpropagation method to compute gradients.

In [3]:
class MSELoss:
    def __init__(self):
        self.gradients = {}
    
    def __call__(self, y_pred, y_true):
        return 0.5 * np.mean((y_true - y_pred) ** 2)
    
    def backpropagate(self, network, X, y_true):
        m = X.shape[0]
        # Output layer gradient
        delta2 = -(y_true - network.a2) * network.sigmoid_derivative(network.a2)
        # Gradients for W2 and b2
        self.gradients['dW2'] = (network.a1.T @ delta2) / m
        self.gradients['db2'] = np.sum(delta2, axis=0, keepdims=True) / m
        # Hidden layer gradient
        delta1 = (delta2 @ network.W2.T) * network.sigmoid_derivative(network.a1)
        # Gradients for W1 and b1
        self.gradients['dW1'] = (X.T @ delta1) / m
        self.gradients['db1'] = np.sum(delta1, axis=0, keepdims=True) / m

## 3. SGD Optimizer Class

Stochastic Gradient Descent optimizer that updates network weights.

In [4]:
class SGDOptimizer: # this is fake SGD (not stochastic for this simplistic data)
    def __init__(self, learning_rate=0.5):
        self.lr = learning_rate
    
    def step(self, network, gradients):
        # Update weights and biases
        network.W2 -= self.lr * gradients['dW2']
        network.b2 -= self.lr * gradients['db2']
        network.W1 -= self.lr * gradients['dW1']
        network.b1 -= self.lr * gradients['db1']

## 4. Prepare XOR Dataset

The XOR problem is a classic test for neural networks because it's not linearly separable.

In [5]:
# XOR dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

print("XOR Truth Table:")
print("-" * 30)
for i in range(len(X)):
    print(f"Input: {X[i]} --> Expected Output: {y[i][0]}")

XOR Truth Table:
------------------------------
Input: [0 0] --> Expected Output: 0
Input: [0 1] --> Expected Output: 1
Input: [1 0] --> Expected Output: 1
Input: [1 1] --> Expected Output: 0


## 5. Initialize Network, Loss, and Optimizer

In [6]:
# Initialize network, loss, and optimizer
network = NeuralNetwork(input_size=2, hidden_size=3, output_size=1)
criterion = MSELoss()
optimizer = SGDOptimizer(learning_rate=0.5)

print("Network initialized with:")
print(f"  Input size: 2")
print(f"  Hidden size: 3")
print(f"  Output size: 1")
print(f"  Learning rate: 0.5")

Network initialized with:
  Input size: 2
  Hidden size: 3
  Output size: 1
  Learning rate: 0.5


## 6. Training Loop

Train the network for 10,000 epochs.

In [7]:
print("Training Neural Network on XOR Problem")
print("=" * 50)

# Training loop
for epoch in range(10000):
    # Forward pass
    predictions = network.forward(X)
    loss = criterion(predictions, y)
    
    # Backward pass. This part is only computing the gradients
    criterion.backpropagate(network, X, y)
    
    # Update weights
    optimizer.step(network, criterion.gradients)
    
    # Print progress
    if (epoch + 1) % 2000 == 0:
        print(f"Epoch {epoch + 1:5d}, Loss: {loss:.6f}")

print("\n" + "-" * 50)
print("Training Complete!")
print("=" * 50)

Training Neural Network on XOR Problem
Epoch  2000, Loss: 0.123078
Epoch  4000, Loss: 0.036327
Epoch  6000, Loss: 0.003731
Epoch  8000, Loss: 0.001581
Epoch 10000, Loss: 0.000961

--------------------------------------------------
Training Complete!


## 7. Evaluate Final Results

In [8]:
# Final predictions
print("\nFinal Results:")
print("-" * 50)
for i in range(len(X)):
    print(f"Input: {X[i]} --> Output: {predictions[i][0]:.4f} (Expected: {y[i][0]})")

print("\n" + "=" * 50)
print("Interpretation:")
print("-" * 50)
print("Values close to 0.0 represent False/0")
print("Values close to 1.0 represent True/1")
print("\nThe network has successfully learned the XOR function!")


Final Results:
--------------------------------------------------
Input: [0 0] --> Output: 0.0423 (Expected: 0)
Input: [0 1] --> Output: 0.9571 (Expected: 1)
Input: [1 0] --> Output: 0.9570 (Expected: 1)
Input: [1 1] --> Output: 0.0470 (Expected: 0)

Interpretation:
--------------------------------------------------
Values close to 0.0 represent False/0
Values close to 1.0 represent True/1

The network has successfully learned the XOR function!


## 8. Get the Network Weights (Optional)

In [9]:
print("\nLearned Weights:")
print("=" * 50)
print("\nW1 (Input --> Hidden):")
print(network.W1)
print("\nb1 (Hidden bias):")
print(network.b1)
print("\nW2 (Hidden --> Output):")
print(network.W2)
print("\nb2 (Output bias):")
print(network.b2)


Learned Weights:

W1 (Input --> Hidden):
[[-5.8938538   2.13778518 -3.30138533]
 [-5.85790486  2.08348754 -3.34480994]]

b1 (Hidden bias):
[[ 2.37222166 -3.24501818  5.08043003]]

W2 (Hidden --> Output):
[[-9.05814492]
 [-4.48851298]
 [ 6.19033535]]

b2 (Output bias):
[[-0.8178816]]


## Regression Example with Parametric Rectified Linear Unit Activations

In [10]:
def train_test_split(x,y,test_size=0.2):
    ind = np.random.permutation(range(len(x)))
    bins = np.array_split(ind,1//test_size)
    testidx = bins[0]
    trainidx = np.delete(range(len(x)),testidx)
    return x[trainidx], x[testidx], y[trainidx], y[testidx]

In [11]:
class PReLU:
    def __init__(self, input_size):
        # Initialize alpha parameters (one per neuron)
        self.alpha = np.ones((1, input_size)) * 0.25
        
    def forward(self, z):
        self.z = z
        return np.where(z > 0, z, self.alpha * z)
    
    def backward(self, grad_output):
        # Gradient w.r.t input
        grad_z = np.where(self.z > 0, 1, self.alpha) * grad_output
        
        # Gradient w.r.t alpha (for updating learnable parameter)
        grad_alpha = np.where(self.z > 0, 0, self.z) * grad_output
        grad_alpha = np.sum(grad_alpha, axis=0, keepdims=True)
        
        return grad_z, grad_alpha


class NeuralNetworkPReLU:
    def __init__(self, input_size, hidden_size, output_size):
        # Initialize weights with He initialization (good for ReLU-like activations)
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))
        
        # PReLU activation for hidden layer
        # note that this is an object based on the PReLU
        self.prelu = PReLU(hidden_size)
    
    def forward(self, X):
        # Hidden layer with PReLU
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.prelu.forward(self.z1)
        
        # Output layer (linear activation for regression)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.z2  # No activation function for regression output
        
        return self.a2
    
    def predict(self, X):
        return self.forward(X)


class MSELoss:
    def __init__(self):
        self.gradients = {}
    
    def __call__(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)
    
    def backpropagate(self, network, X, y_true):
        m = X.shape[0]
        
        # Output layer gradient (derivative of MSE + linear activation)
        delta2 = (network.a2 - y_true) / m
        
        # Gradients for output layer
        self.gradients['dW2'] = network.a1.T @ delta2
        self.gradients['db2'] = np.sum(delta2, axis=0, keepdims=True)
        
        # Backpropagate through PReLU
        delta1_pre = delta2 @ network.W2.T
        delta1, grad_alpha = network.prelu.backward(delta1_pre)
        
        # Gradients for hidden layer
        self.gradients['dW1'] = X.T @ delta1
        self.gradients['db1'] = np.sum(delta1, axis=0, keepdims=True)
        
        # Gradient for PReLU alpha parameter
        self.gradients['d_alpha'] = grad_alpha


class SGDOptimizer: # (Again this is not stochastic)
    def __init__(self, learning_rate=0.01):
        self.lr = learning_rate
    
    def step(self, network, gradients):
        network.W2 -= self.lr * gradients['dW2']
        network.b2 -= self.lr * gradients['db2']
        network.W1 -= self.lr * gradients['dW1']
        network.b1 -= self.lr * gradients['db1']
        
        # Update PReLU alpha parameter
        network.prelu.alpha -= self.lr * gradients['d_alpha']


def load_concrete_data():

    url = "https://github.com/dvasiliu/AML/blob/main/Data%20Sets/concrete.csv?raw=true"
    df = pd.read_csv(url)
    
    # Extract features and target
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1:].values

    return X, y


def train_network(network, criterion, optimizer, X_train, y_train, 
                  X_test, y_test, epochs=6000, print_every=500):
    train_losses = []
    test_losses = []
    
    for epoch in range(epochs):
        # Forward pass
        predictions = network.forward(X_train)
        train_loss = criterion(predictions, y_train)
        
        # Backward pass
        criterion.backpropagate(network, X_train, y_train)
        
        # Update parameters
        optimizer.step(network, criterion.gradients)
        
        # Test loss
        test_predictions = network.predict(X_test)
        test_loss = criterion(test_predictions, y_test)
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        
        # Print progress
        if (epoch + 1) % print_every == 0:
            print(f"Epoch {epoch + 1:5d} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")
    
    return train_losses, test_losses


def calculate_metrics(y_true, y_pred):
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    
    # R-squared
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }



print("=" * 70)
print("Neural Network for Concrete Compressive Strength Prediction")
print("Using PReLU Activation Function")
print("=" * 70)
print()

# Load data
print("Loading Concrete Compressive Strength Dataset...")
X, y = load_concrete_data()
print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Target statistics: mean={y.mean():.2f}, std={y.std():.2f} MPa")
print()

# Split data:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print()

# Normalize features only (NOT target values)
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

print("Data normalized using StandardScaler (features only)")
print()

# Initialize network
input_size = X_train.shape[1]  # 8 features
hidden_size = 20  # Number of neurons in hidden layer
output_size = 1   # Single output for regression

network = NeuralNetworkPReLU(
    input_size=input_size,
    hidden_size=hidden_size,
    output_size=output_size
)

print(f"Network Architecture:")
print(f"  Input layer:  {input_size} neurons")
print(f"  Hidden layer: {hidden_size} neurons (PReLU activation)")
print(f"  Output layer: {output_size} neuron (Linear activation)")
print(f"  Total parameters: {input_size * hidden_size + hidden_size + hidden_size * output_size + output_size + hidden_size}")
print()

# Initialize loss and optimizer
criterion = MSELoss()
optimizer = SGDOptimizer(learning_rate=0.01)

# Train the network
print("Training Started...")
print("-" * 70)

train_losses, test_losses = train_network(
    network, criterion, optimizer,
    X_train, y_train, X_test, y_test,
    epochs=5000,
    print_every=500
)

print("-" * 70)
print("Training Complete!")
print()

# Evaluate on test set
print("=" * 70)
print("Model Evaluation on Test Set")
print("=" * 70)
print()

test_predictions = network.predict(X_test)

metrics = calculate_metrics(y_test, test_predictions)

print(f"Mean Squared Error (MSE):  {metrics['MSE']:.4f}")
print(f"Root Mean Squared Error (RMSE): {metrics['RMSE']:.4f} MPa")
print(f"Mean Absolute Error (MAE): {metrics['MAE']:.4f} MPa")
print(f"R2 Score: {metrics['R2']:.4f}")
print()

# Show learned PReLU alpha values
print("=" * 70)
print("Learned PReLU Alpha Parameters")
print("=" * 70)
print()
print("Alpha values (one per hidden neuron):")
print(network.prelu.alpha)
print(f"\nMean alpha: {network.prelu.alpha.mean():.4f}")
print(f"Std alpha:  {network.prelu.alpha.std():.4f}")
print()

# Show sample predictions
print("=" * 70)
print("Sample Predictions")
print("=" * 70)
print()
n_samples = min(10, len(X_test))
print(f"{'Predicted':<12} {'Actual':<12} {'Error':<12}")
print("-" * 36)
for i in range(n_samples):
    pred = test_predictions[i, 0]
    actual = y_test[i, 0]
    error = pred - actual
    print(f"{pred:<12.2f} {actual:<12.2f} {error:<12.2f}")

print()
print("=" * 70)
print("Training Summary")
print("=" * 70)
print()
print(f"Final Training Loss:   {train_losses[-1]:.6f}")
print(f"Final Test Loss: {test_losses[-1]:.6f}")
print(f"Test RMSE: {metrics['RMSE']:.2f} MPa")

Neural Network for Concrete Compressive Strength Prediction
Using PReLU Activation Function

Loading Concrete Compressive Strength Dataset...
Dataset shape: X=(1030, 8), y=(1030, 1)
Target statistics: mean=35.82, std=16.70 MPa

Train set: 772 samples
Test set: 258 samples

Data normalized using StandardScaler (features only)

Network Architecture:
  Input layer:  8 neurons
  Hidden layer: 20 neurons (PReLU activation)
  Output layer: 1 neuron (Linear activation)
  Total parameters: 221

Training Started...
----------------------------------------------------------------------
Epoch   500 | Train Loss: 27.8732 | Test Loss: 44.2986
Epoch  1000 | Train Loss: 22.0074 | Test Loss: 37.4345
Epoch  1500 | Train Loss: 20.5216 | Test Loss: 37.0895
Epoch  2000 | Train Loss: 20.6803 | Test Loss: 38.1840
Epoch  2500 | Train Loss: 20.6875 | Test Loss: 39.2648
Epoch  3000 | Train Loss: 20.6703 | Test Loss: 39.5730
Epoch  3500 | Train Loss: 20.4708 | Test Loss: 39.7062
Epoch  4000 | Train Loss: 20.114